# 00 – Veri hazırlama ve hasta-düzeyinde bölme

Bu notebook **tüm modeller için tek ve ortak** bölmeyi üretir (`splits/train|val|test.csv`).
Çalıştırma dizini: `vindr/` ve `mimic/` klasörlerinin bulunduğu dizin (eski `dataset_bilgileri.ipynb` ile aynı).

* MIMIC-CXR: **subject_id** bazlı bölme (hasta ve çalışma sızıntısı yok; `assert_no_leak` doğrular)
* VinDr-CXR: hasta kimliği yok → görüntü düzeyinde (makalede sınırlılık olarak belirtilmeli)

In [ ]:
import sys, os; sys.path.insert(0, "src")
import numpy as np, pandas as pd
import cxr_data as D

DATA_ROOT = "."          # vindr/ ve mimic/ klasörlerinin bulunduğu dizin
SPLIT_DIR = "splits"
RES_DIR   = "results"
os.makedirs(RES_DIR, exist_ok=True)

In [ ]:
# Tüm veriyi birleştir. vindr_agg='max' eski kodla aynıdır (3 radyolog birleşimi);
# 'majority' (>=2 radyolog) daha tutucudur — makalede hangisini kullandığınızı belirtin.
df_all, label_cols = D.build_master(DATA_ROOT, vindr_agg="max", frontal_only=False)
print(label_cols)

In [ ]:
sp = D.make_splits(df_all, label_cols, seed=D.SEED, mimic_test=7500, vindr_test=2500, val_frac=0.10)
display(D.leakage_report(sp))
D.save_splits(sp, SPLIT_DIR)

## Sızıntı doğrulaması (yeni bölme)

In [ ]:
mim_tr = pd.concat([sp.train, sp.val]).query("source=='mimic'")
mim_te = sp.test.query("source=='mimic'")
print("test hastası eğitim/doğrulamada olan :", f"{mim_te.subject.isin(set(mim_tr.subject)).mean():.1%}")
print("test çalışması eğitim/doğrulamada olan:", f"{mim_te.study.isin(set(mim_tr.study)).mean():.1%}")

## Karşılaştırma: eski satır-düzeyi bölmenin sızıntısı (yaklaşık yeniden üretim)

In [ ]:
from sklearn.model_selection import train_test_split
mim = df_all[df_all.source == "mimic"]
a, b = train_test_split(mim, test_size=7500, random_state=42)          # eski mantık: satır düzeyi
print("ESKİ tarz bölme — test hastası eğitimde olan :", f"{b.subject.isin(set(a.subject)).mean():.1%}")
print("ESKİ tarz bölme — test çalışması eğitimde olan:", f"{b.study.isin(set(a.study)).mean():.1%}")
print("(Eski kodun gerçek ölçümü: %97,2 hasta, %67,5 çalışma. Burada tabakalama yok, bu yüzden yaklaşık.)")

## Etiket sayıları (bölme başına, kaynak başına) — nadir sınıfları belirlemek için

In [ ]:
rows = []
for name, d in (("train", sp.train), ("val", sp.val), ("test", sp.test)):
    for src, s in d.groupby("source"):
        for c in label_cols:
            v = s[c].dropna()
            rows.append(dict(split=name, source=src, disease=c, n_defined=len(v), n_pos=int(v.sum())))
cnt = pd.DataFrame(rows)
cnt.to_csv(f"{RES_DIR}/label_counts_by_split.csv", index=False)
piv = cnt.pivot_table(index="disease", columns=["split", "source"], values="n_pos", aggfunc="sum").fillna(0).astype(int)
display(piv)
t = cnt[cnt.split == "test"].groupby("disease").n_pos.sum().sort_values()
print("Test'te pozitif sayısı < 30 olan (AUC güvenilmez) hastalıklar:")
print(t[t < 30].to_string())

## Sabit değerlendirme ızgarası (özet)

Test ızgarası = her test görüntüsü × o kaynakta etiketi **tanımlı** her hastalık (`No finding` sorulmaz).
Eski koddaki rastgele soru üretimi kaldırıldı; böylece Full/Noise/Black/Shuffle ve modeller arası karşılaştırmalar **aynı çiftler üzerinde eşleşiktir**.

In [ ]:
g = D.build_eval_grid(sp.test, label_cols)
print("test çifti:", len(g), "| görüntü:", g.img_idx.nunique())
display(D.grid_summary(g))